# AstroRAG — AI Agents

Atividades neste notebook:

- construção de agentes científicos;
- implementação de ferramentas (tools);
- adicionamento de reasoning workflows;
- criação de um sistema agentic para FRBs.

In [1]:
import pickle
import numpy as np
import faiss
import ollama

from sentence_transformers import SentenceTransformer

### Carregando Chunks

In [2]:
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

texts = [chunk.page_content for chunk in chunks]

print(f"Chunks carregados: {len(texts)}")

Chunks carregados: 1763


### Embeddings

In [3]:
embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

### FAISS

In [4]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print(f"Vetores indexados: {index.ntotal}")

Vetores indexados: 1763


### Tool: Retrieval

In [5]:
def retrieve_context(query, k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    contexts = []

    for idx in indices[0]:
        contexts.append(texts[idx])

    return "\n\n".join(contexts)

### Tool: Summarizer

In [6]:
def summarize_context(context):

    prompt = f"""
Summarize the scientific context below.

Context:
{context}

Summary:
"""

    response = ollama.chat(
        model='mistral',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )

    return response['message']['content']

### Tool: Concept Explainer

In [7]:
def explain_concept(concept):

    context = retrieve_context(concept)

    prompt = f"""
You are a scientific assistant specialized in radio astronomy.

Explain the following concept clearly.

Concept:
{concept}

Scientific Context:
{context}

Explanation:
"""

    response = ollama.chat(
        model='mistral',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )

    return response['message']['content']

### Agent reasoning

In [8]:
def decide_tool(query):

    query_lower = query.lower()

    if "summarize" in query_lower:
        return "summarizer"

    elif "explain" in query_lower:
        return "explainer"

    elif "what is" in query_lower:
        return "explainer"

    else:
        return "retrieval"

### AstroAgent

In [9]:
def astroagent(query):

    tool = decide_tool(query)

    print(f"Tool selecionada: {tool}")

    if tool == "summarizer":

        context = retrieve_context(query)

        return summarize_context(context)

    elif tool == "explainer":

        return explain_concept(query)

    else:

        context = retrieve_context(query)

        prompt = f"""
Answer the question using the scientific context below.

Context:
{context}

Question:
{query}

Answer:
"""

        response = ollama.chat(
            model='mistral',
            messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ]
        )

        return response['message']['content']

# Testando...

### Teste 1

In [10]:
query = "What are Fast Radio Bursts?"

response = astroagent(query)

print(response)

Tool selecionada: retrieval
 Fast Radio Bursts (FRBs) are millisecond-duration bursts that originate from distant parts of the universe and can be detected in radio waves. They have remained a significant puzzle in astrophysics due to their mysterious origins, short duration, and high dispersion measures. These bursts are subject to intense observational and theoretical investigations, aiming to unravel their underlying physics.


### Teste 2

In [12]:
query = "Explain dispersion measure in FRBs"

response = astroagent(query)

print(response)

Tool selecionada: explainer
 Dispersion Measure (DM) in Fast Radio Bursts (FRBs) refers to a crucial parameter that measures the amount of delay experienced by different radio frequencies due to the plasma through which the radio waves are passing. This delay arises because higher frequency radio waves travel slower than lower frequency ones when they move through an ionized medium, such as a plasma.

The DM is significant in FRBs as it helps us understand the environment and distance of the source emitting these mysterious bursts. The higher the DM, the denser the ionized medium between us (observers) and the FRB source. In other words, DM provides information about the column density of free electrons along the line of sight from the FRB to Earth.

To calculate the amount of delay caused by dispersion measure, we can use the equation ∆tDM = (8.3 µs) DM*(∆νMHz)^-3, where ∆tDM is the time delay in microseconds, DM is the dispersion measure in units of pc cm^-3, and ∆ν is the frequency 

### Teste 3

In [13]:
query = "Summarize how CHIME detects Fast Radio Bursts"

response = astroagent(query)

print(response)

Tool selecionada: summarizer
 The scientific context revolves around the CHIME (Canadian Hydrogen Intensity Mapping Experiment) instrument, specifically its use for detecting Fast Radio Bursts (FRBs). The CHIME/FRB project searches for FRBs in real-time using high time and frequency resolution data within CHIME's field of view.

The text discusses the backend of the CHIME/FRB system, including its real-time FRB search and detection software pipeline, as well as planned offline analyses. The predicted detection rate for CHIME/FRB is 0.6 to 11 bursts per day, revised from earlier estimates based on a Euclidean flux distribution and assuming an intrinsic width of 3 ms for the FRBs with scattering timescales drawn from a Galactic scattering-time distribution at 1 GHz. This new prediction is presented in Figure 8 and was obtained using simulations similar to those used by Chawla et al. (2017).

Additional references are provided for further reading on the topic, including works by Wang et a

### Teste 4

In [14]:
query = "How CHIME differs from traditional radio telescopes?"

response = astroagent(query)

print(response)

Tool selecionada: retrieval
 CHIME (The Canadian Hydrogen Intensity Mapping Experiment) differs from traditional radio telescopes in several key aspects:

1. Collecting Area and Focal Ratio: CHIME has a much larger collecting area of 8000 m² compared to most traditional radio telescopes, which allows it to gather more signal from distant sources. Its focal ratio (f/D) of 0.25 is also quite low, making it more efficient at collecting light from extended sources like the sky itself rather than point sources like stars.

2. Wide Field of View (FoV): CHIME has a significantly wider field of view compared to traditional radio telescopes. In the east-west direction, it covers 2.5 degrees to 1.3 degrees, and in the north-south direction, it spans approximately 110 degrees. This wide FoV allows CHIME to survey large areas of the sky quickly.

3. Polarization: Instead of focusing on a single polarization state like traditional radio telescopes often do, CHIME is equipped with orthogonal linear 

In [15]:
query = "Explain how CHIME differs from traditional radio telescopes"

response = astroagent(query)

print(response)

Tool selecionada: explainer
 CHIME (the Canadian Hydrogen Intensity Mapping Experiment) is a novel type of radio telescope that differs significantly from traditional radio telescopes, especially in terms of its design, functionality, and observational capabilities. Here are some key aspects where CHIME stands out:

1. Wide Field of View (FoV): Traditional radio telescopes often have a narrow field of view to achieve high resolution and sensitivity in a limited area. In contrast, CHIME boasts an extremely wide field of view, spanning approximately 110 degrees in the north-south direction and 2.5 degrees to -1.3 degrees in the east-west direction. This allows CHIME to observe a much larger portion of the sky at once, making it highly efficient for surveys and transient event detection.

2. Focal Plane Arrangement: Instead of using individual antennas with a central receiver or focusing optics like traditional telescopes, CHIME employs an array of 1024 dual-polarization feed horns evenly

### Teste 5

In [16]:
query = "Summarize the main characteristics of Fast Radio Bursts"

response = astroagent(query)

print(response)

Tool selecionada: summarizer
 The article by Bing Zhang, published in September 2023, discusses the ongoing investigation into Fast Radio Bursts (FRBs), mysterious radio signals that last for mere milliseconds and are believed to originate from extragalactic or even cosmological distances. Although much research has been conducted in recent years, their origin remains a significant puzzle in contemporary astrophysics.

The article cites several studies, including one from 2026 by Ailton J. B. Júnior et al., which focuses on the application of machine learning techniques for feature selection and classification of FRBs. Another study mentioned is from 2016 by P. Scholz et al., which reports on multi-wavelength observations and additional bursts of the repeating FRB 121102.

The article also includes acknowledgements and an appendix, discussing libraries used in the research and their reproducibility. The main focus remains on understanding the origin of these intriguing cosmic phenomena

### Teste 6

In [17]:
query = "How can machine learning improve FRB classification?"

response = astroagent(query)

print(response)

Tool selecionada: retrieval
 Machine learning improves Fast Radio Bursts (FRB) classification by providing a means to classify FRBs into repeaters and non-repeaters with reasonable accuracy. Various supervised machine learning methods, such as random forests, logistic regression, gradient boosting, and shallow neural networks, have been applied to FRB catalog datasets to achieve this. However, these approaches lack interpretability. Some FRBs identified as repeater candidates in previous literature are false positives from different methodological approaches, like FRB20181218C, which was identified by a variety of supervised machine learning methods with a relatively low confidence score. Clustering algorithms can also be used to identify FRBs but may require additional features. Additionally, the use of machine-learning algorithms in the CHIME/FRB pipeline helps improve RFI mitigation, real-time event identification, and offline event detection, demonstrating the power of data-rich so